# Backtest protocol (audit-ready, executable)

**As of (UTC):** 2026-05-05  
This notebook implements the protocol described in `backtest_protocol.md` using deposited CSV artefacts.


In [ ]:
import csv, math, os, random, subprocess, sys
from pathlib import Path

ROOT = Path('/var/www/clients/client1/web3/web')
DEPOSIT = ROOT / 'research' / 'deposit'
BACKTEST = DEPOSIT / 'backtest'
BACKTEST_NAMED = DEPOSIT / 'backtest-named'

EVENTS_CSV = BACKTEST / 'backtest_events.csv.txt'
NAMED_CSV = BACKTEST_NAMED / 'named_events_backtest.csv.txt'

assert EVENTS_CSV.exists(), EVENTS_CSV
assert NAMED_CSV.exists(), NAMED_CSV

print('OK: inputs found')
print(' -', EVENTS_CSV)
print(' -', NAMED_CSV)


In [ ]:
def read_csv_dict(path: Path):
    rows = []
    with path.open(newline='') as f:
        r = csv.DictReader(f)
        for row in r:
            rows.append(row)
    return rows

events = read_csv_dict(EVENTS_CSV)
named = read_csv_dict(NAMED_CSV)

print('event-style rows:', len(events))
print('named-event rows:', len(named))
print('sample event row:', events[0])


In [ ]:
# Metrics: delta, z, and a simple effect-size proxy.
def f(x):
    return float(x) if x not in (None, '') else float('nan')

deltas = [f(r['delta']) for r in events]
zs = [f(r['z_vs_pre']) for r in events]

def median(xs):
    ys = sorted(xs)
    n = len(ys)
    if n == 0:
        return float('nan')
    return ys[n//2] if n % 2 == 1 else 0.5*(ys[n//2-1]+ys[n//2])

abs_delta_median = median([abs(x) for x in deltas])
frac_z_ge_2 = sum(1 for z in zs if abs(z) >= 2.0) / max(1, len(zs))

print('median |delta|:', round(abs_delta_median, 4))
print('fraction |z|>=2:', round(frac_z_ge_2, 4))


In [ ]:
# Significance: permutation test for mean shift, using pre_mean vs event-day score.
# We do not have full windows in the deposited summary CSV, but we can still test
# whether event-day scores differ from pre_mean across events (paired difference).

diffs = []
for r in events:
    score = f(r['score'])
    pre_mean = f(r['pre_mean'])
    diffs.append(score - pre_mean)

obs = sum(diffs) / max(1, len(diffs))

random.seed(1337)
N = 10000
more_extreme = 0
for _ in range(N):
    # sign-flip permutation (paired): under null, diff sign is symmetric
    s = 0.0
    for d in diffs:
        s += d if (random.random() < 0.5) else (-d)
    stat = s / max(1, len(diffs))
    if abs(stat) >= abs(obs):
        more_extreme += 1

p = (more_extreme + 1) / (N + 1)

print('paired mean(score - pre_mean):', round(obs, 6))
print('permutation p-value (two-sided, sign-flip):', p)


## Sensitivity on the fixture (±10% per connector)

This section computes a delta-matrix on the synthetic fixture by re-running the deterministic PHP recompute.


In [ ]:
RECOMPUTE = DEPOSIT / 'recompute' / 'recompute_nfsi_fixture.php'
FIXTURE_DIR = DEPOSIT / 'sample-dataset'
META = FIXTURE_DIR / 'connector_meta.csv.txt'

assert RECOMPUTE.exists(), RECOMPUTE
assert META.exists(), META

def run_recompute(out_path: Path, extra_env=None):
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    cmd = [sys.executable.replace('python', 'php'), str(RECOMPUTE), f'--fixture-dir={FIXTURE_DIR}', f'--out={out_path}']
    # fallback if sys.executable trick fails
    if not Path(cmd[0]).exists():
        cmd[0] = 'php'
    subprocess.check_call(cmd, env=env)

baseline_csv = Path('/tmp/nfsi_fixture_baseline.csv')
run_recompute(baseline_csv)
print('baseline written:', baseline_csv)


In [ ]:
# Read baseline (iso2,date)->nfsi
def read_nfsi_map(path: Path):
    m = {}
    with path.open(newline='') as f:
        r = csv.DictReader(f)
        for row in r:
            key = (row['iso2'], row['date_ymd'])
            m[key] = float(row['nfsi_today'])
    return m

base_map = read_nfsi_map(baseline_csv)
print('baseline keys:', len(base_map))


In [ ]:
# Sensitivity implementation note:
# The PHP recompute reads weights from the fixture's connector_meta.csv.txt.
# For OAT perturbations we create a temporary modified copy of that CSV and point the recompute at it.

def load_meta_rows():
    with META.open(newline='') as f:
        r = csv.DictReader(f)
        rows = list(r)
    return r.fieldnames, rows

fields, meta_rows = load_meta_rows()
connector_ids = [r['connector_id'] for r in meta_rows if r.get('connector_id')]
print('connectors:', connector_ids)

tmp_dir = Path('/tmp/nfsi_sensitivity')
tmp_dir.mkdir(parents=True, exist_ok=True)

def write_meta(rows, path: Path):
    with path.open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            w.writerow(r)

def run_with_meta(meta_path: Path, out_path: Path):
    # Copy fixture dir with overridden meta file only
    # (recompute uses --fixture-dir, so we build a minimal temp fixture dir)
    fx = tmp_dir / meta_path.stem
    fx.mkdir(parents=True, exist_ok=True)
    # copy required inputs
    for name in ['connectors_raw.csv.txt', 'country_meta.csv.txt', 'nfsi_prev.csv.txt']:
        src = FIXTURE_DIR / name
        dst = fx / name
        dst.write_bytes(src.read_bytes())
    (fx / 'connector_meta.csv.txt').write_bytes(meta_path.read_bytes())
    cmd = ['php', str(RECOMPUTE), f'--fixture-dir={fx}', f'--out={out_path}']
    subprocess.check_call(cmd)

delta_matrix = []
for cid in connector_ids:
    perturbed = []
    for r in meta_rows:
        rr = dict(r)
        if rr['connector_id'] == cid:
            base_w = float(rr['connector_weight'])
            rr['connector_weight'] = str(base_w * 1.10)
        perturbed.append(rr)
    meta_path = tmp_dir / f'meta_{cid}_plus10.csv'
    write_meta(perturbed, meta_path)
    out_csv = tmp_dir / f'out_{cid}_plus10.csv'
    run_with_meta(meta_path, out_csv)
    m = read_nfsi_map(out_csv)
    # store deltas
    for (iso2, d), v in m.items():
        base = base_map[(iso2, d)]
        delta_matrix.append({'iso2': iso2, 'date_ymd': d, 'perturb': f'{cid}_plus10', 'delta_nfsi': round(v-base, 4)})

print('delta rows:', len(delta_matrix))


In [ ]:
# Write sensitivity delta matrix as deposit artefact
out_path = BACKTEST / 'sensitivity_fixture_delta_matrix.csv.txt'
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open('w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['iso2','date_ymd','perturb','delta_nfsi'])
    w.writeheader()
    for r in delta_matrix:
        w.writerow(r)

print('WROTE:', out_path)
